In [1]:
# %% Cell: THE CHAMPION BASELINE (Score: 0.8032)
# Use this code to study the "winning" logic.

import pandas as pd
import numpy as np
import os
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
import copy

# 1. SETUP & DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🏆 RESTORING CHAMPION BASELINE on {device}...")

# 2. LOAD DATA
if os.path.exists('/kaggle/input'):
    search_path = '/kaggle/input'
    for root, dirs, files in os.walk(search_path):
        if 'train_v2.csv' in files:
            DATA_PATH = root; break
else:
    DATA_PATH = '../data/raw'

print("   📂 Loading Raw Data...")
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train_v2.csv'))
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_v2.csv'))

# Sorting is critical for time-series lags
train_df = train_df.sort_values(['date_id', 'minute_id']).reset_index(drop=True)
test_df = test_df.sort_values(['date_id', 'minute_id']).reset_index(drop=True)

# 3. FEATURE ENGINEERING (The "Smoothing" Logic)
print("   ⚙️ Engineering Standard Features...")
target_cols = ['target_short', 'target_medium', 'target_long']
ignore = ['id', 'date_id', 'minute_id'] + target_cols
base_features = [c for c in train_df.columns if c not in ignore]
vip_features = ['feature_19', 'feature_5', 'feature_27', 'feature_2', 'feature_13']

def engineer_standard(df):
    df_eng = df.copy()
    
    # A. Lags (Context from the past)
    for col in base_features:
        for lag in [1, 2, 3, 5]:
            df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
            
    # B. Rolling Means (The Low-Pass Filter)
    # This removes the "noise" and keeps the "trend"
    for w in [5, 10, 20]:
        for col in vip_features:
            df_eng[f'{col}_mean_{w}'] = df_eng.groupby('date_id')[col].transform(lambda x: x.rolling(w).mean())
            df_eng[f'{col}_std_{w}'] = df_eng.groupby('date_id')[col].transform(lambda x: x.rolling(w).std())
            
    # C. Ranks (Relative Strength)
    for col in vip_features:
        df_eng[f'{col}_rank'] = df_eng.groupby('date_id')[col].rank(pct=True) - 0.5
        
    return df_eng

train_eng = engineer_standard(train_df)
y = train_eng[target_cols].values
test_eng = engineer_standard(test_df)

del train_df, test_df
gc.collect()

# 4. PREPARE & SCALE (The "Robust" Logic)
final_cols = [c for c in train_eng.columns if c not in ignore]
X = train_eng[final_cols].values
X_test = test_eng[final_cols].values

# Force-Clean NaNs (NNs hate NaNs)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)
y = np.nan_to_num(y, nan=0.0)

print("   ⚖️ Robust Scaling (Handling Outliers)...")
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# 5. CLUSTERING (The "Context" Logic)
print("   🧩 Generating 7 Market States (The Sweet Spot)...")
all_data = np.vstack([X_scaled, X_test_scaled])
kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
kmeans.fit(all_data[::10]) 
train_clusters = kmeans.predict(X_scaled)
test_clusters = kmeans.predict(X_test_scaled)

# 6. MODEL SETUP
print("   🧠 Configuring 5-Seed Ensemble...")

SEEDS = [42, 43, 44, 101, 777]
N_FOLDS = 5
EPOCHS = 20
BATCH_SIZE = 1024 
LEARNING_RATE = 1e-3
TARGET_SCALE = 100.0

# GPU Tensors
X_num_tensor = torch.FloatTensor(X_scaled).to(device)
X_test_num_tensor = torch.FloatTensor(X_test_scaled).to(device)
cluster_train_tensor = torch.LongTensor(train_clusters).to(device)
cluster_test_tensor = torch.LongTensor(test_clusters).to(device)
y_tensor = torch.FloatTensor(y * TARGET_SCALE).to(device)

# --- THE CHAMPION ARCHITECTURE ---
# A Dual-Stream Network:
# Stream 1: Processes Math (Numbers)
# Stream 2: Processes Meaning (Clusters)
class EmbeddingMLP(nn.Module):
    def __init__(self, num_inputs, hidden_dim=256, output_dim=3):
        super().__init__()
        
        # STREAM 1: Embedding (The Context)
        self.embedding = nn.Embedding(7, 4) 
        
        # STREAM 2: Numerical (The Math)
        self.num_layer = nn.Sequential(
            nn.Linear(num_inputs, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.3)
        )
        
        # MERGE: Glue them together
        self.combined = nn.Sequential(
            nn.Linear(hidden_dim + 4, hidden_dim), # Input + Embed Dim
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, x_num, x_cat):
        emb = self.embedding(x_cat) 
        num = self.num_layer(x_num)
        concat = torch.cat([num, emb], dim=1) # The Fusion happens here
        return self.combined(concat)

# 7. TRAINING LOOP
ensemble_preds = np.zeros((len(X_test), 3))

for seed in SEEDS:
    print(f"\n   🌱 Seed {seed}...")
    kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    seed_preds = np.zeros((len(X_test), 3))
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_num_tensor, y_tensor)):
        
        train_ds = TensorDataset(X_num_tensor[train_idx], cluster_train_tensor[train_idx], y_tensor[train_idx])
        val_ds = TensorDataset(X_num_tensor[val_idx], cluster_train_tensor[val_idx], y_tensor[val_idx])
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
        
        model = EmbeddingMLP(num_inputs=X_num_tensor.shape[1]).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
        
        # Huber Loss: The reason this model beat L1 Loss
        # It handles noise better than L1, but outliers better than MSE.
        criterion = nn.HuberLoss()
        
        best_loss = float('inf')
        best_weights = copy.deepcopy(model.state_dict())
        
        for epoch in range(EPOCHS):
            model.train()
            for bx_num, bx_cat, by in train_loader:
                optimizer.zero_grad()
                out = model(bx_num, bx_cat)
                loss = criterion(out, by)
                loss.backward()
                optimizer.step()
                
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for bx_num, bx_cat, by in val_loader:
                    out = model(bx_num, bx_cat)
                    val_loss += criterion(out, by).item()
            
            avg_loss = val_loss / len(val_loader)
            if avg_loss < best_loss:
                best_loss = avg_loss
                best_weights = copy.deepcopy(model.state_dict())
        
        # Predict Fold
        model.load_state_dict(best_weights)
        model.eval()
        with torch.no_grad():
            fold_p = []
            test_loader = DataLoader(TensorDataset(X_test_num_tensor, cluster_test_tensor), batch_size=4096)
            for bx_num, bx_cat in test_loader:
                fold_p.append(model(bx_num, bx_cat).cpu().numpy())
            seed_preds += np.concatenate(fold_p) / N_FOLDS
        
        del model, optimizer; torch.cuda.empty_cache()
    
    ensemble_preds += seed_preds / len(SEEDS)
    print(f"   ✅ Seed {seed} Done.")

# 8. SUBMISSION
ensemble_preds = ensemble_preds / TARGET_SCALE
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_v2.csv'), usecols=['id'])

sub = pd.DataFrame({
    'id': test_df['id'],
    'target_short': ensemble_preds[:, 0],
    'target_medium': ensemble_preds[:, 1],
    'target_long': ensemble_preds[:, 2]
})

for c in sub.columns[1:]:
    sub[c] = sub[c] - sub[c].mean()

sub.to_csv('../submissions/submission_champion_baseline.csv', index=False)
print("🚀 Saved 'submission_champion_baseline.csv'")

🏆 RESTORING CHAMPION BASELINE on cpu...
   📂 Loading Raw Data...
   ⚙️ Engineering Standard Features...


/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_19978/754665623.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_19978/754665623.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_19978/754665623.py:51: PerformanceWarning: DataFrame is highly fragmented.  Thi

   ⚖️ Robust Scaling (Handling Outliers)...
   🧩 Generating 7 Market States (The Sweet Spot)...
   🧠 Configuring 5-Seed Ensemble...

   🌱 Seed 42...
   ✅ Seed 42 Done.

   🌱 Seed 43...
   ✅ Seed 43 Done.

   🌱 Seed 44...
   ✅ Seed 44 Done.

   🌱 Seed 101...
   ✅ Seed 101 Done.

   🌱 Seed 777...
   ✅ Seed 777 Done.
🚀 Saved 'submission_champion_baseline.csv'
